### 1. Importação de Bibliotecas

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, Tuple
from PIL import Image
from tqdm import tqdm
import time

### 2. Definição de funções utilitárias

Esta célula define duas funções utilitárias:

- `extract_id_from_filename`
    -  Extrai um identificador numérico de um nome de arquivo.
- `calculate_overlap_area`
    -  Calcula a área de interseção entre dois retângulos.

In [ ]:
def extract_id_from_filename(filename: str) -> Optional[int]:
    try:
        return int(filename.split('.')[0])
    except ValueError:
        return None

def calculate_overlap_area(box1: Tuple[float, float, float, float], box2: Tuple[float, float, float, float]) -> float:
    x1_1, y1_1, x1_2, y1_2 = box1
    x2_1, y2_1, x2_2, y2_2 = box2
    
    x_left = max(x1_1, x2_1)
    y_top = max(y1_1, y2_1)
    x_right = min(x1_2, x2_2)
    y_bottom = min(y1_2, y2_2)
    
    if x_right < x_left or y_bottom < y_top:
        return 0.0
    
    return (x_right - x_left) * (y_bottom - y_top)

### 3. Definição da função responsável pelo mapa de calor

Esta célula de código define uma função crucial para a análise de layouts de documentos. Sua finalidade é gerar um mapa de calor (heatmap) que visualiza a densidade e sobreposição de ilustrações em uma página.

A função opera em três etapas principais:

1. **Preparação e Filtragem de Dados**
    - Valida os dados de entrada.
    - *(Opcional)* Filtra as ilustrações com base em um intervalo de datas.
2. **Rasterização de Polígonos**
    - Converte as coordenadas de cada ilustração em uma grade de pixels.
    - Para cada "pixel" na grade, ela conta quantas ilustrações se sobrepõem naquela área, acumulando essa contagem. Este processo cria uma "grade de sobreposição" numérica.
3. **Criação e Exibição do Gráfico**
    - Transforma a grade de sobreposição em um heatmap visual.
    - O gráfico pode ser exibido diretamente ou salvo como um arquivo de imagem.
   
Em resumo, transformamos dados de localização de ilustrações em uma representação visual que destaca padrões de layout e áreas de maior concentração de imagens.

In [ ]:
def create_illustration_overlap_map(
        illustration_data: pd.DataFrame,
        page_width: int = 1825, page_height: int = 2170, resolution: int = 10,
        date_range: Optional[Tuple[str, str]] = None,
        save_figure: bool = False, filename: str = "overlap.png",
        title: str = "Mapa de Calor"
) -> None:

    # 1. Preparação e Filtragem dos Dados
    print("1. Preparando os dados...")
    if date_range:
        if 'date' not in illustration_data.columns:
            raise ValueError("O DataFrame deve conter uma coluna 'date' quando date_range é fornecido.")
        
        try:
            # Converte a coluna 'date' para datetime e filtra o DataFrame
            illustration_data['date'] = pd.to_datetime(illustration_data['date'], format='%Y-%m-%d', errors='raise')
            start_date, end_date = pd.to_datetime(date_range[0]), pd.to_datetime(date_range[1])
            illustration_data = illustration_data[
                (illustration_data['date'] >= start_date) & (illustration_data['date'] <= end_date)
            ].copy()
        except ValueError as e:
            raise ValueError(f"Formato da coluna 'date' inválido. Detalhes: {e}")

    x1_coords, y1_coords, x2_coords, y2_coords = (illustration_data[col].values for col in ['bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2'])
    num_illustrations = len(x1_coords)
    print(f"Número de ilustrações: {num_illustrations}")

    # 2. Rasterização dos Polígonos (Criação da Grade de Sobreposição)
    # Define dimensões da grade com base na resolução
    cols, rows = int(page_width * resolution), int(page_height * resolution)
    overlap_grid = np.zeros((rows, cols), dtype=np.uint16) # Grade de contagem de sobreposição
    page_area = page_width * page_height # Calcula a área total da página uma vez

    for i in tqdm(range(num_illustrations), desc="Rasterizando ilustrações"):
        # Extrai coordenadas da ilustração atual
        x1, y1, x2, y2 = x1_coords[i], y1_coords[i], x2_coords[i], y2_coords[i]

        # Converte coordenadas da bounding box para coordenadas de pixel na grade
        # A coordenada Y é invertida para alinhamento com a origem 'lower' do imshow
        left, right = int(x1 * resolution), int(x2 * resolution)
        bottom, top = int((page_height - y2) * resolution), int((page_height - y1) * resolution)

        # Garante que as coordenadas estejam dentro dos limites da grade
        left, top = max(0, left), max(0, top)
        right, bottom = min(cols, right), min(rows, bottom)

        # Se houver uma área válida para incrementar, adiciona 1 às células correspondentes
        if left < right and bottom < top:
            overlap_grid[bottom:top, left:right] += 1

    # 3. Criação e Exibição do Gráfico (Heatmap)
    print("3. Criando o gráfico...")
    plt.figure(figsize=(4, 5))
    ax = plt.gca()
    ax.set(title=title, xlim=(0, page_width), ylim=(0, page_height))

    # Exibe a grade rasterizada como um heatmap
    img = ax.imshow(overlap_grid, cmap='hot',
                    extent=[0, page_width, 0, page_height], # Mapeia as dimensões da grade para as da página
                    interpolation='none', # Exibe pixels discretos, sem suavização
                    aspect='auto',        # Ajusta a proporção automaticamente
                    origin='lower')       # Define a origem da grade no canto inferior esquerdo

    # Adiciona barra de cores como legenda
    plt.colorbar(img, ax=ax)
    
    print("4. Exibindo ou salvando o gráfico...")
    if save_figure:
        # Salva a figura
        plt.savefig(filename, bbox_inches='tight')
        print(f"Figura salva em {filename}")

    plt.show() # Exibe o gráfico

### 4. Definição do fluxo principal.

Agora, organizamos o fluxo de trabalho do programa. A função que definiremos será responsável por:

- Carregar os metadados das ilustrações e das datas.
- Mesclar esses dados.
- Iterar sobre um conjunto de páginas para chamar a função definida na célula anterior `create_illustration_overlap_map`, gerando um heatmap para cada página.

In [ ]:
def main():
    try:
        df_illustrations = pd.read_csv('cropped_images_metadata.csv')
        df_dates = pd.read_csv('records_with_images_metadata.csv')
    except FileNotFoundError:
        print("Erro: Um ou ambos os arquivos CSV não foram encontrados. Verifique se os caminhos estão corretos.")
        return

    df_illustrations['page_id'] = df_illustrations['original_filename'].apply(extract_id_from_filename)
    df_merged = pd.merge(df_illustrations, df_dates, on='page_id', how='inner')

    if 'date' not in df_merged.columns:
        raise ValueError("O DataFrame mesclado não contém uma coluna “date”. Verifique as chaves de mesclagem.")
    
    for page_number in range(1, 9):
        page_data = df_merged[
            (df_merged['page'] == page_number) | # (1) Linhas onde a página original é o número atual do loop (1-8)
            (
                (df_merged['page'] > 8) & # (2a) Linhas onde a página original é um "almanaque" (maior que 8)
                ((df_merged['page'] - 1) % 8 + 1) == page_number # (2b) E essa página de almanaque, se normalizada, resulta no número atual do loop
            )
        ].copy()
        
        # Chamar a função com os dados filtrados
        print(f"Processando dados para a página: {page_number}")
        create_illustration_overlap_map(page_data, date_range=('1855-01-01', '1860-12-31'), save_figure=True, filename=f"MP{page_number}.png", title=f"Página {page_number}")
    
    if df_merged.empty:
        print("O DataFrame final está vazio. Verifique seus dados e parâmetros.")

Essa célula simplesmente chama a função main(), que dá início a todo o processo definido nas células anteriores.

In [ ]:
main()